In [32]:
import numpy as np

text = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
chars = sorted(list(set(text))) # 去重后转换成列表并排序
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
seq_len = 3

In [33]:
# Generate dataset for training
X, y = [], []
for i in range(len(text) - seq_len):
    input = text[i:i+seq_len]
    output = text[i+seq_len]
    X.append([char2idx[c] for c in input])
    y.append(char2idx[output])
X = np.array(X)
y = np.eye(vocab_size)[y]

In [34]:
# Initiate parameters
hidden_dim = 128
Wxh = np.random.randn(vocab_size, hidden_dim) * 0.01
Whh = np.random.randn(hidden_dim, hidden_dim) * 0.01
Why = np.random.randn(hidden_dim, vocab_size) * 0.01
bh = np.zeros((1, hidden_dim))
by = np.zeros((1, vocab_size))

In [35]:
# Forward function
def forward(x, h_prev):
    h = []
    h.append(h_prev)
    for t in range(seq_len):
        x_t = x[t].reshape(1, -1)
        h_t = np.tanh(np.dot(x_t, Wxh) + np.dot(h[t], Whh) + bh)
        h.append(h_t)

    y_pred = np.dot(h[-1], Why) + by
    return y_pred, h[-1]

In [36]:
# Train model
learning_rate = 0.005
epochs = 1000
h_prev = np.zeros((1, hidden_dim))

for epoch in range(epochs):
    total_loss = 0
    h_prev = np.zeros((1, hidden_dim))

    for i in range(len(X)):
        x_onehot = np.eye(vocab_size)[X[i]]
        y_pred, h_t = forward(x_onehot, h_prev)

        loss = -np.log(y_pred[0][np.argmax(y[i])])
        total_loss += loss
        
        # Updating parameters in hand
        dy = y_pred - y[i].reshape(1, -1)
        dWhy = np.dot(h_t.T, dy)
        dby = np.sum(dy, axis=0, keepdims=True)
        dh = np.dot(dy, Why.T) * (1 - h_t ** 2)
        dWhh = np.dot(h_prev.T, dh)
        x_last = x_onehot[-1].reshape(-1, 1)
        dWxh = np.dot(x_last, dh)
        dbh = np.sum(dh, axis=0, keepdims=True)

        Wxh -= learning_rate * dWxh
        Whh -= learning_rate * dWhh
        Why -= learning_rate * dWhy
        bh -= learning_rate * dbh
        by -= learning_rate * dby

        h_prev = h_t

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(X):.2f}")

/tmp/ipykernel_20725/530170530.py:14: RuntimeWarning: invalid value encountered in log
  loss = -np.log(y_pred[0][np.argmax(y[i])])


Epoch 100, Loss: 3.43
Epoch 200, Loss: 2.79
Epoch 300, Loss: 2.02
Epoch 400, Loss: 1.23
Epoch 500, Loss: 0.57
Epoch 600, Loss: 0.19
Epoch 700, Loss: 0.05
Epoch 800, Loss: 0.01
Epoch 900, Loss: 0.00
Epoch 1000, Loss: 0.00


In [37]:
# Generate text
def generate_text(seed_text, gen_len=5):
    '''
    gen_len is the length of generation
    '''
    h_prev = np.zeros((1, hidden_dim))

    seed_idx = [char2idx[c] for c in seed_text]
    generated = seed_text
    for _ in range(gen_len):
        x_onehot = np.eye(vocab_size)[seed_idx]
        y_pred, h_prev = forward(x_onehot, h_prev)
        next_idx = np.argmax(y_pred)
        generated += idx2char[next_idx]

        seed_idx = seed_idx[1:] + [next_idx]
    return generated

In [39]:
# test
seed = "abc"
print("生成文本：", generate_text(seed, gen_len=5))

生成文本： abcdefgh
